# 44 - Smoke test SPIDER desde GCS

Notebook de prueba controlada para validar que el flujo de entrenamiento sagital puede consumir el dataset SPIDER migrado a Google Cloud Storage. Es un smoke test funcional, no un entrenamiento final ni una medicion de calidad.

## Garantias del flujo

- Solo lee objetos desde GCS: lista, consulta metadata y descarga muestras seleccionadas.
- No sube, borra, reescribe, compone ni modifica metadata en el bucket.
- Entrena exactamente una epoca sobre 6 pares SPIDER deterministas.
- Guarda el checkpoint y la evidencia solo en Google Drive.
- El checkpoint generado queda marcado como `smoke_test`, no apto para produccion.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED_MODULES = {
    "google.cloud.storage": "google-cloud-storage",
    "google_crc32c": "google-crc32c",
    "pandas": "pandas",
    "numpy": "numpy",
    "PIL": "pillow",
    "SimpleITK": "SimpleITK",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "matplotlib": "matplotlib",
}
missing = [pkg for module, pkg in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *sorted(set(missing))])

import base64
import csv
import hashlib
import json
import math
import os
import random
import tempfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import google.auth
import google_crc32c
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn.functional as F
from google.cloud import storage
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset

## Repositorio y arquitectura

Clona o actualiza el AI Module en Colab y usa exclusivamente `SagittalUNet2D` del repositorio.

In [ ]:
PFI_REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
PFI_REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")

def run_git(args: list[str], cwd: Path | None = None) -> str:
    completed = subprocess.run(["git", *args], cwd=str(cwd) if cwd else None, check=True, text=True, capture_output=True)
    return completed.stdout.strip()

if not PFI_REPO_ROOT.exists():
    run_git(["clone", PFI_REPO_URL, str(PFI_REPO_ROOT)])
else:
    run_git(["fetch"], cwd=PFI_REPO_ROOT)
    try:
        run_git(["pull", "--ff-only"], cwd=PFI_REPO_ROOT)
    except subprocess.CalledProcessError as exc:
        print("No se pudo actualizar con fast-forward; se usa el checkout local existente.")

MODEL_ARCH_PATH = PFI_REPO_ROOT / "ai_service" / "pfi_ai_service" / "model_architectures.py"
if not MODEL_ARCH_PATH.exists():
    raise FileNotFoundError(f"Falta model_architectures.py en {MODEL_ARCH_PATH}")
sys.path.insert(0, str(PFI_REPO_ROOT / "ai_service"))
from pfi_ai_service.model_architectures import SagittalUNet2D

REPOSITORY_COMMIT = run_git(["rev-parse", "HEAD"], cwd=PFI_REPO_ROOT)
print(json.dumps({"repository_commit": REPOSITORY_COMMIT}, indent=2))

## Configuracion

Parametros congelados para que el smoke test sea chico, reproducible y barato.

In [ ]:
@dataclass(frozen=True)
class SmokeConfig:
    PROJECT_ID: str = "pfi-asplanatti-fabrello-v1"
    BUCKET_NAME: str = "pfi-rm-lumbar-artifacts-2026-ef"
    PRELIGHT_DIR: Path = Path("/content/drive/MyDrive/PFI_MVP/results/GCS_training_preflight")
    INDEX_CSV: Path = PRELIGHT_DIR / "spider_training_index.csv"
    PREFLIGHT_SUMMARY_JSON: Path = PRELIGHT_DIR / "gcs_training_preflight_summary.json"
    PREFLIGHT_EVIDENCE_JSON: Path = PRELIGHT_DIR / "gcs_training_preflight_evidence.json"
    OUTPUT_DIR: Path = Path("/content/drive/MyDrive/PFI_MVP/results/GCS_spider_smoke_test")
    LOCAL_CACHE_DIR: Path = Path("/content/pfi_gcs_spider_smoke")
    SEED: int = 20260721
    NUM_SELECTED_PAIRS: int = 6
    TRAIN_PAIRS: int = 4
    VAL_PAIRS: int = 1
    TEST_PAIRS: int = 1
    TARGET_SIZE: tuple[int, int] = (256, 256)
    NUM_CLASSES: int = 4
    BASE_CHANNELS: int = 8
    BATCH_SIZE: int = 2
    MAX_EPOCHS: int = 1
    MAX_SLICES_PER_VOLUME: int = 4
    MAX_BACKGROUND_SLICES_PER_VOLUME: int = 1
    MAX_TRAIN_BATCHES: int = 8
    MAX_VAL_BATCHES: int = 4
    MAX_TEST_BATCHES: int = 4
    LEARNING_RATE: float = 0.001
    NUM_WORKERS: int = 0
    SMOKE_MODE: bool = True
    ALLOW_GCS_WRITE: bool = False

CFG = SmokeConfig()
assert CFG.SMOKE_MODE is True
assert CFG.ALLOW_GCS_WRITE is False
assert CFG.MAX_EPOCHS == 1
assert CFG.NUM_SELECTED_PAIRS == 6
assert CFG.TRAIN_PAIRS + CFG.VAL_PAIRS + CFG.TEST_PAIRS == 6
assert CFG.MAX_TRAIN_BATCHES <= 8
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
print(CFG)

## Preflight validado

Carga el indice y la evidencia del notebook 43 desde Drive. El flujo se detiene si el preflight no fue exitoso o si el SHA no coincide.

In [ ]:
def ensure_drive_available() -> None:
    drive_root = Path("/content/drive")
    my_drive = drive_root / "MyDrive"
    if my_drive.exists():
        return
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Google Drive no esta montado y no se detecta Colab.") from exc
    drive.mount(str(drive_root))


def require_file(path: Path) -> None:
    if not path.exists() or not path.is_file():
        raise FileNotFoundError(path)


def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_json(path: Path) -> dict[str, Any]:
    require_file(path)
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)

ensure_drive_available()
for required_path in [CFG.INDEX_CSV, CFG.PREFLIGHT_SUMMARY_JSON, CFG.PREFLIGHT_EVIDENCE_JSON]:
    require_file(required_path)
preflight_summary = read_json(CFG.PREFLIGHT_SUMMARY_JSON)
preflight_evidence = read_json(CFG.PREFLIGHT_EVIDENCE_JSON)
EXPECTED_MANIFEST_SHA256 = "fa54c89a278d850021c0f91c0a27b3b5211c86301c9e4f125d75d517f39b793b"
assert preflight_summary.get("TRAINING_PREFLIGHT_SUCCESS") is True
assert preflight_summary.get("manifest_sha256") == EXPECTED_MANIFEST_SHA256
assert int(preflight_summary.get("spider_complete_pairs")) == 447
assert int(preflight_summary.get("expected_objects_verified")) == 2058
assert int(preflight_summary.get("unexpected_objects")) == 0
assert int(preflight_summary.get("missing_objects")) == 0
assert int(preflight_summary.get("sampled_spider_pairs_opened")) == 3
training_index_sha256 = sha256_stream(CFG.INDEX_CSV)
expected_index_sha = preflight_evidence.get("spider_training_index_sha256")
if training_index_sha256 != expected_index_sha:
    raise ValueError("SHA del indice SPIDER no coincide con la evidencia del preflight")
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({"preflight_success_confirmed": True, "training_index_sha256": training_index_sha256}, indent=2))

## Indice y cliente GCS read-only

Valida el indice SPIDER de 447 pares y prepara un cliente de Storage solo para lectura.

In [ ]:
INDEX_COLUMNS = ["pair_id", "image_gcs_uri", "mask_gcs_uri", "image_size_bytes", "mask_size_bytes", "image_crc32c", "mask_crc32c"]
GCS_WRITE_OPERATIONS = 0

def validate_spider_index(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, keep_default_na=False)
    missing_cols = [col for col in INDEX_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Faltan columnas en indice SPIDER: {missing_cols}")
    df = df[INDEX_COLUMNS].copy()
    if len(df) != 447:
        raise ValueError(f"Cantidad inesperada de pares SPIDER: {len(df)}")
    if df["pair_id"].astype(str).str.strip().eq("").any() or df["pair_id"].duplicated().any():
        raise ValueError("pair_id vacio o duplicado")
    for col in ["image_gcs_uri", "mask_gcs_uri"]:
        if df[col].duplicated().any() or not df[col].astype(str).str.startswith(f"gs://{CFG.BUCKET_NAME}/").all():
            raise ValueError(f"URI invalida o duplicada en {col}")
    for col in ["image_size_bytes", "mask_size_bytes"]:
        df[col] = pd.to_numeric(df[col], errors="raise").astype("int64")
        if (df[col] <= 0).any():
            raise ValueError(f"Tamanios no positivos en {col}")
    for col in ["image_crc32c", "mask_crc32c"]:
        if df[col].astype(str).str.strip().eq("").any():
            raise ValueError(f"CRC32C vacio en {col}")
    return df.sort_values("pair_id", kind="stable").reset_index(drop=True)


def authenticate_storage_client() -> storage.Client:
    try:
        from google.colab import auth
        auth.authenticate_user()
    except ImportError:
        pass
    credentials, auth_project = google.auth.default()
    print(json.dumps({"auth_project_detected": auth_project, "client_project": CFG.PROJECT_ID}, indent=2))
    return storage.Client(project=CFG.PROJECT_ID, credentials=credentials)


def object_name_from_uri(uri: str) -> str:
    prefix = f"gs://{CFG.BUCKET_NAME}/"
    if not uri.startswith(prefix):
        raise ValueError("URI fuera del bucket esperado")
    return uri[len(prefix):]

spider_index_df = validate_spider_index(CFG.INDEX_CSV)
storage_client = authenticate_storage_client()
bucket = storage_client.bucket(CFG.BUCKET_NAME)
bucket.reload()
if bucket.name != CFG.BUCKET_NAME:
    raise RuntimeError("Bucket inesperado")

## Seleccion y descarga

Selecciona 6 pares deterministas, con pacientes distintos, y descarga 12 objetos verificando tamano y CRC32C.

In [ ]:
def patient_id_from_pair(pair_id: str) -> str:
    return str(pair_id).replace("-", "_").split("_")[0]


def select_six_pairs(index_df: pd.DataFrame) -> pd.DataFrame:
    candidates = []
    positions = np.linspace(0, len(index_df) - 1, CFG.NUM_SELECTED_PAIRS, dtype=int).tolist()
    seen_patients = set()
    for pos in positions + list(range(len(index_df))):
        row = index_df.iloc[int(pos)]
        patient_id = patient_id_from_pair(row["pair_id"])
        if patient_id in seen_patients:
            continue
        item = row.to_dict()
        item["selected_position"] = len(candidates)
        item["patient_id"] = patient_id
        item["split"] = "train" if len(candidates) < CFG.TRAIN_PAIRS else ("val" if len(candidates) == CFG.TRAIN_PAIRS else "test")
        candidates.append(item)
        seen_patients.add(patient_id)
        if len(candidates) == CFG.NUM_SELECTED_PAIRS:
            break
    if len(candidates) != CFG.NUM_SELECTED_PAIRS:
        raise RuntimeError("No se consiguieron seis pacientes distintos para smoke test")
    return pd.DataFrame(candidates)

selected_pairs_df = select_six_pairs(spider_index_df)
selected_pairs_df.to_csv(CFG.OUTPUT_DIR / "selected_smoke_pairs.csv", index=False)
print(json.dumps({"selected_pairs": len(selected_pairs_df), "train_pairs": CFG.TRAIN_PAIRS, "val_pairs": CFG.VAL_PAIRS, "test_pairs": CFG.TEST_PAIRS}, indent=2))


def full_suffix(path_or_uri: str) -> str:
    name = Path(object_name_from_uri(path_or_uri)).name.lower()
    if name.endswith(".nii.gz"):
        return ".nii.gz"
    return Path(name).suffix


def crc32c_base64(path: Path) -> str:
    checksum = google_crc32c.Checksum()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            checksum.update(chunk)
    return base64.b64encode(checksum.digest()).decode("ascii")


def download_checked(uri: str, expected_size: int, expected_crc32c: str, target: Path) -> Path:
    blob = bucket.blob(object_name_from_uri(uri))
    blob.reload()
    blob.download_to_filename(str(target))
    if not target.exists() or target.stat().st_size <= 0:
        raise RuntimeError("Descarga vacia")
    if target.stat().st_size != int(expected_size):
        raise RuntimeError("Tamanio local no coincide con indice")
    if crc32c_base64(target) != str(expected_crc32c):
        raise RuntimeError("CRC32C local no coincide con indice")
    return target

download_records = []
for row in selected_pairs_df.itertuples(index=False):
    sample_dir = CFG.LOCAL_CACHE_DIR / f"sample_{int(row.selected_position):03d}"
    sample_dir.mkdir(parents=True, exist_ok=True)
    image_path = download_checked(row.image_gcs_uri, row.image_size_bytes, row.image_crc32c, sample_dir / f"image{full_suffix(row.image_gcs_uri)}")
    mask_path = download_checked(row.mask_gcs_uri, row.mask_size_bytes, row.mask_crc32c, sample_dir / f"mask{full_suffix(row.mask_gcs_uri)}")
    download_records.append({"selected_position": int(row.selected_position), "split": row.split, "patient_id": row.patient_id, "pair_id": row.pair_id, "image_local_path": str(image_path), "mask_local_path": str(mask_path)})
downloaded_objects = len(download_records) * 2
verified_downloads = downloaded_objects
if downloaded_objects != 12:
    raise RuntimeError("Cantidad inesperada de descargas")
print(json.dumps({"downloaded_objects": downloaded_objects, "verified_downloads": verified_downloads}, indent=2))

## Lectura medica y mapping

Abre `.npy`, imagenes comunes, `.mha/.mhd` con SimpleITK y `.nii/.nii.gz` con nibabel bajo demanda. Aplica el mapping sagital SPIDER al espacio de 4 clases.

In [ ]:
def open_medical_array(path: Path) -> np.ndarray:
    suffixes = [s.lower() for s in path.suffixes]
    suffix = path.suffix.lower()
    if suffix == ".npy":
        return np.load(path)
    if suffix in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
        return np.asarray(Image.open(path))
    if suffix in {".mha", ".mhd"}:
        image = sitk.ReadImage(str(path))
        return sitk.GetArrayFromImage(image)
    if suffix == ".nii" or suffixes[-2:] == [".nii", ".gz"]:
        if importlib.util.find_spec("nibabel") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "nibabel"])
        import nibabel as nib
        return np.asarray(nib.load(str(path)).get_fdata())
    raise RuntimeError(f"Formato medico no soportado: {suffix}")


def canonicalize_spider_array(arr):
    arr = np.asarray(arr)
    if arr.ndim == 3 and arr.shape[0] <= 64 and arr.shape[-1] > 64:
        return np.moveaxis(arr, 0, -1)
    return arr


def normalize_label_mapping(raw: Any) -> dict[int, int]:
    if isinstance(raw, dict) and "label_group_mapping" in raw:
        raw = raw["label_group_mapping"]
    mapping: dict[int, int] = {}
    for key, value in raw.items():
        if isinstance(value, list):
            class_id = int(key)
            for raw_label in value:
                mapping[int(raw_label)] = class_id
        else:
            mapping[int(key)] = int(value)
    if not mapping:
        raise ValueError("Mapping sagital vacio")
    bad_classes = sorted({v for v in mapping.values() if v < 0 or v >= CFG.NUM_CLASSES})
    if bad_classes:
        raise ValueError(f"Mapping produce clases fuera de 0..3: {bad_classes}")
    return mapping


def load_json_mapping(path: Path) -> dict[int, int]:
    with path.open("r", encoding="utf-8") as fh:
        return normalize_label_mapping(json.load(fh))


def load_label_mapping() -> dict[int, int]:
    mapping_json = Path("/content/drive/MyDrive/PFI_MVP/results/E5_multiclase_agrupado/E5_multiclass_label_mapping.json")
    checkpoint_path = Path("/content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best.pt")
    if mapping_json.exists():
        return load_json_mapping(mapping_json)
    if checkpoint_path.exists():
        checkpoint = torch.load(
        str(checkpoint_path),
        map_location="cpu",
        weights_only=False,
    )
        if "label_group_mapping" not in checkpoint:
            raise RuntimeError("Checkpoint sagital sin label_group_mapping")
        return normalize_label_mapping(checkpoint["label_group_mapping"])
    raise FileNotFoundError("No se encontro mapping sagital real")


def apply_label_map(mask: np.ndarray, label_map: dict[int, int]) -> np.ndarray:
    labels = sorted(int(v) for v in np.unique(mask))
    missing = [label for label in labels if label not in label_map]
    if missing:
        raise ValueError(f"Etiqueta cruda sin mapping: {missing[:10]}")
    out = np.zeros(mask.shape, dtype=np.int64)
    for raw_label, class_id in label_map.items():
        out[mask == raw_label] = class_id
    if out.min() < 0 or out.max() >= CFG.NUM_CLASSES:
        raise ValueError("Mascara mapeada fuera de 0..3")
    return out

label_group_mapping = load_label_mapping()
with (CFG.OUTPUT_DIR / "smoke_label_group_mapping.json").open("w", encoding="utf-8") as fh:
    json.dump({str(k): int(v) for k, v in sorted(label_group_mapping.items())}, fh, indent=2)

## Slices reales

Canonicaliza SPIDER, selecciona slices sagitales con foreground cuando existan y congela el CSV de slices usados.

In [ ]:
SPIDER_SAGITTAL_AXIS = 2

def choose_indices(indices: np.ndarray, limit: int) -> list[int]:
    if len(indices) == 0:
        return []
    if len(indices) <= limit:
        return [int(v) for v in indices]
    positions = np.linspace(0, len(indices) - 1, limit, dtype=int)
    return [int(indices[pos]) for pos in positions]

slice_rows = []
for record in download_records:
    image_arr = canonicalize_spider_array(open_medical_array(Path(record["image_local_path"])))
    raw_mask_arr = canonicalize_spider_array(open_medical_array(Path(record["mask_local_path"])))
    if image_arr.size <= 0 or raw_mask_arr.size <= 0 or image_arr.ndim not in {2, 3} or raw_mask_arr.ndim not in {2, 3} or image_arr.shape != raw_mask_arr.shape:
        raise ValueError("Volumen SPIDER invalido")
    if not np.isfinite(image_arr).all():
        raise ValueError("Imagen SPIDER contiene valores no finitos")
    mask_arr = apply_label_map(raw_mask_arr, label_group_mapping)
    unique_mask_values = np.unique(mask_arr)
    if len(unique_mask_values) == 0:
        raise ValueError("Mascara sin valores unicos")
    if image_arr.ndim == 3 and image_arr.shape[SPIDER_SAGITTAL_AXIS] <= 0:
        raise ValueError("Eje sagital 2 inexistente")
    if image_arr.ndim == 2:
        candidate = [(0, bool((mask_arr > 0).any()))]
    else:
        fg = np.array([idx for idx in range(mask_arr.shape[SPIDER_SAGITTAL_AXIS]) if (np.take(mask_arr, idx, axis=SPIDER_SAGITTAL_AXIS) > 0).any()])
        bg = np.array([idx for idx in range(mask_arr.shape[SPIDER_SAGITTAL_AXIS]) if not (np.take(mask_arr, idx, axis=SPIDER_SAGITTAL_AXIS) > 0).any()])
        candidate = [(idx, True) for idx in choose_indices(fg, CFG.MAX_SLICES_PER_VOLUME)]
        candidate += [(idx, False) for idx in choose_indices(bg, CFG.MAX_BACKGROUND_SLICES_PER_VOLUME)]
    for slice_index, has_foreground in candidate:
        unique_values = np.unique(mask_arr)
        print(json.dumps({
            "sample_position": int(record["selected_position"]),
            "image_shape": list(image_arr.shape),
            "image_dtype": str(image_arr.dtype),
            "mask_shape": list(mask_arr.shape),
            "mask_dtype": str(mask_arr.dtype),
            "mask_unique_values_first_30": [int(v) for v in unique_values[:30]],
            "mask_unique_count": int(len(unique_values)),
        }, indent=2))
        slice_rows.append({"split": record["split"], "sample_position": record["selected_position"], "patient_id": record["patient_id"], "image_local_path": record["image_local_path"], "mask_local_path": record["mask_local_path"], "slice_index": int(slice_index), "slice_axis": SPIDER_SAGITTAL_AXIS if image_arr.ndim == 3 else 0, "has_foreground": bool(has_foreground)})

slice_records_df = pd.DataFrame(slice_rows)
if slice_records_df.empty:
    raise RuntimeError("No se generaron slices para smoke test")
for split in ["train", "val", "test"]:
    subset = slice_records_df[slice_records_df["split"] == split]
    if subset.empty or not bool(subset["has_foreground"].any()):
        raise RuntimeError(f"Split {split} sin slices foreground")
slice_records_df.to_csv(CFG.OUTPUT_DIR / "smoke_slice_records.csv", index=False)
print(json.dumps({"train_slices": int((slice_records_df.split == 'train').sum()), "val_slices": int((slice_records_df.split == 'val').sum()), "test_slices": int((slice_records_df.split == 'test').sum())}, indent=2))

## Dataset y tensores

Preprocesa a 1x256x256 con normalizacion robusta y valida batches de train, val y test.

In [ ]:
def robust_normalize(image: np.ndarray) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    lo, hi = np.percentile(arr, [1, 99])
    if hi <= lo:
        return np.zeros(arr.shape, dtype=np.float32)
    arr = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
    return arr.astype(np.float32)


def extract_slice(arr: np.ndarray, slice_index: int, axis: int) -> np.ndarray:
    if arr.ndim == 2:
        return arr
    return np.take(arr, slice_index, axis=axis)


def resize_image_2d(image: np.ndarray) -> np.ndarray:
    scaled = (robust_normalize(image) * 255).astype(np.uint8)
    pil = Image.fromarray(scaled).resize((CFG.TARGET_SIZE[1], CFG.TARGET_SIZE[0]), Image.Resampling.BILINEAR)
    return (np.asarray(pil).astype(np.float32) / 255.0)[None, :, :]


def resize_mask_2d(mask: np.ndarray) -> np.ndarray:
    pil = Image.fromarray(mask.astype(np.int32), mode="I").resize((CFG.TARGET_SIZE[1], CFG.TARGET_SIZE[0]), Image.Resampling.NEAREST)
    return np.asarray(pil, dtype=np.int64)

@dataclass(frozen=True)
class SmokeSliceRecord:
    split: str
    sample_position: int
    patient_id: str
    image_local_path: str
    mask_local_path: str
    slice_index: int
    slice_axis: int
    has_foreground: bool

class SpiderSmokeDataset(Dataset):
    def __init__(self, records: list[SmokeSliceRecord], label_map: dict[int, int]):
        self.records = records
        self.label_map = label_map
    def __len__(self) -> int:
        return len(self.records)
    def __getitem__(self, idx: int):
        rec = self.records[idx]
        image_arr = canonicalize_spider_array(open_medical_array(Path(rec.image_local_path)))
        mask_arr = apply_label_map(canonicalize_spider_array(open_medical_array(Path(rec.mask_local_path))), self.label_map)
        image_slice = extract_slice(image_arr, rec.slice_index, rec.slice_axis)
        mask_slice = extract_slice(mask_arr, rec.slice_index, rec.slice_axis)
        image = torch.from_numpy(resize_image_2d(image_slice)).float()
        mask = torch.from_numpy(resize_mask_2d(mask_slice)).long()
        return image, mask

records = [SmokeSliceRecord(**row) for row in slice_records_df.to_dict("records")]
loaders = {}
for split in ["train", "val", "test"]:
    split_records = [r for r in records if r.split == split]
    loaders[split] = DataLoader(SpiderSmokeDataset(split_records, label_group_mapping), batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)

def validate_batch(name: str, batch):
    image, mask = batch
    if image.ndim != 4 or mask.ndim != 3 or image.shape[1] != 1 or image.shape[-2:] != torch.Size(CFG.TARGET_SIZE) or mask.shape[-2:] != torch.Size(CFG.TARGET_SIZE):
        raise RuntimeError(f"Batch {name} con shape invalida")
    if not torch.isfinite(image).all() or mask.min() < 0 or mask.max() >= CFG.NUM_CLASSES:
        raise RuntimeError(f"Batch {name} invalido")

for name, loader in loaders.items():
    validate_batch(name, next(iter(loader)))

## Entrenamiento smoke

Instancia `SagittalUNet2D` con `base_channels=8`, ejecuta un forward real y entrena una unica epoca acotada.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else None
model = SagittalUNet2D(in_channels=1, num_classes=CFG.NUM_CLASSES, base_channels=CFG.BASE_CHANNELS).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(json.dumps({"device": str(device), "gpu_name": gpu_name, "total_params": total_params, "trainable_params": trainable_params}, indent=2))

first_batch = next(iter(loaders["train"]))
with torch.no_grad():
    smoke_logits = model(first_batch[0].to(device))
if smoke_logits.shape != (first_batch[0].shape[0], CFG.NUM_CLASSES, CFG.TARGET_SIZE[0], CFG.TARGET_SIZE[1]) or not torch.isfinite(smoke_logits).all():
    raise RuntimeError("Forward smoke invalido")
smoke_pred = smoke_logits.argmax(dim=1)
if smoke_pred.min() < 0 or smoke_pred.max() >= CFG.NUM_CLASSES:
    raise RuntimeError("Predicciones fuera de rango antes de entrenar")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE)
tracked_param = next(p for p in model.parameters() if p.requires_grad)
tracked_before = tracked_param.detach().clone()
train_history = []
optimizer_steps = 0
model.train()
for epoch in range(CFG.MAX_EPOCHS):
    for batch_index, (image, mask) in enumerate(loaders["train"]):
        if batch_index >= CFG.MAX_TRAIN_BATCHES:
            break
        image = image.to(device)
        mask = mask.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(image)
        loss = criterion(logits, mask)
        if not torch.isfinite(loss):
            raise RuntimeError("Loss de entrenamiento no finita")
        loss.backward()
        optimizer.step()
        optimizer_steps += 1
        train_history.append({"batch_index": int(batch_index), "loss": float(loss.detach().cpu()), "batch_size": int(image.shape[0]), "device": str(device)})
if optimizer_steps <= 0:
    raise RuntimeError("No hubo optimizer steps")
parameter_delta_l2 = float(torch.linalg.vector_norm((tracked_param.detach() - tracked_before).flatten()).cpu())
if parameter_delta_l2 <= 0:
    raise RuntimeError("El parametro monitoreado no cambio")
train_loss = float(np.mean([row["loss"] for row in train_history]))
if not math.isfinite(train_loss):
    raise RuntimeError("Train loss no finita")
pd.DataFrame(train_history).to_csv(CFG.OUTPUT_DIR / "smoke_batch_history.csv", index=False)

## Evaluacion smoke

Evalua validacion y test con Dice por clase como senal funcional. No hay umbral de calidad.

In [ ]:
def dice_by_class(pred: torch.Tensor, target: torch.Tensor, num_classes: int) -> dict[int, float | None]:
    result: dict[int, float | None] = {}
    for cls in range(num_classes):
        pred_mask = pred == cls
        target_mask = target == cls
        denom = int(pred_mask.sum() + target_mask.sum())
        if denom == 0:
            result[cls] = None
        else:
            inter = int((pred_mask & target_mask).sum())
            result[cls] = float((2 * inter) / denom)
    return result


def evaluate_loader(name: str, loader: DataLoader, max_batches: int) -> dict[str, Any]:
    model.eval()
    losses, dices = [], []
    pred_min, pred_max = CFG.NUM_CLASSES - 1, 0
    batches = 0
    with torch.no_grad():
        for batch_index, (image, mask) in enumerate(loader):
            if batch_index >= max_batches:
                break
            image = image.to(device)
            mask = mask.to(device)
            logits = model(image)
            loss = criterion(logits, mask)
            if not torch.isfinite(loss) or not torch.isfinite(logits).all():
                raise RuntimeError(f"Eval {name} no finita")
            pred = logits.argmax(dim=1)
            if pred.min() < 0 or pred.max() >= CFG.NUM_CLASSES:
                raise RuntimeError(f"Predicciones {name} fuera de 0..3")
            pred_min = min(pred_min, int(pred.min().cpu()))
            pred_max = max(pred_max, int(pred.max().cpu()))
            losses.append(float(loss.cpu()))
            dices.append(dice_by_class(pred.cpu(), mask.cpu(), CFG.NUM_CLASSES))
            batches += 1
    if batches <= 0:
        raise RuntimeError(f"Sin batches en {name}")
    by_class = {}
    for cls in range(CFG.NUM_CLASSES):
        values = [d[cls] for d in dices if d[cls] is not None]
        by_class[cls] = None if not values else float(np.mean(values))
    fg_values = [v for cls, v in by_class.items() if cls != 0 and v is not None]
    fg_macro = float(np.mean(fg_values)) if fg_values else None
    return {"loss": float(np.mean(losses)), "dice_by_class": by_class, "foreground_dice": fg_macro, "prediction_min_class": pred_min, "prediction_max_class": pred_max, "batches": batches}

val_result = evaluate_loader("val", loaders["val"], CFG.MAX_VAL_BATCHES)
test_result = evaluate_loader("test", loaders["test"], CFG.MAX_TEST_BATCHES)
for value in [val_result["loss"], test_result["loss"]]:
    if not math.isfinite(float(value)):
        raise RuntimeError("Loss de evaluacion no finita")

## Checkpoint y evidencia

Guarda checkpoint smoke, preview y JSON de evidencia en Drive. El exito exige GCS read-only, descargas verificadas, parametros cambiados y outputs locales generados.

In [ ]:
def save_prediction_preview(path: Path) -> None:
    model.eval()
    image, mask = next(iter(loaders["test"]))
    with torch.no_grad():
        pred = model(image.to(device)).argmax(dim=1).cpu()[0].numpy()
    image_np = image[0, 0].numpy()
    mask_np = mask[0].numpy()
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    axes[0].imshow(image_np, cmap="gray")
    axes[0].set_title("Imagen")
    axes[1].imshow(mask_np, vmin=0, vmax=CFG.NUM_CLASSES - 1)
    axes[1].set_title("Mascara real")
    axes[2].imshow(pred, vmin=0, vmax=CFG.NUM_CLASSES - 1)
    axes[2].set_title("Prediccion")
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=140)
    plt.close(fig)

checkpoint_path = CFG.OUTPUT_DIR / "sagittal_spider_smoke_checkpoint.pt"
preview_path = CFG.OUTPUT_DIR / "smoke_prediction_preview.png"
summary_path = CFG.OUTPUT_DIR / "gcs_spider_smoke_test_summary.json"
evidence_path = CFG.OUTPUT_DIR / "gcs_spider_smoke_test_evidence.json"
selected_pairs_path = CFG.OUTPUT_DIR / "selected_smoke_pairs.csv"
slice_records_path = CFG.OUTPUT_DIR / "smoke_slice_records.csv"
batch_history_path = CFG.OUTPUT_DIR / "smoke_batch_history.csv"
mapping_path = CFG.OUTPUT_DIR / "smoke_label_group_mapping.json"

checkpoint = {
    "model_key": "sagittal_spider_smoke",
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": 1,
    "num_classes": CFG.NUM_CLASSES,
    "base_channels": CFG.BASE_CHANNELS,
    "target_size": list(CFG.TARGET_SIZE),
    "seed": CFG.SEED,
    "selected_pairs": CFG.NUM_SELECTED_PAIRS,
    "train_slices": int((slice_records_df.split == "train").sum()),
    "val_slices": int((slice_records_df.split == "val").sum()),
    "test_slices": int((slice_records_df.split == "test").sum()),
    "label_group_mapping": {str(k): int(v) for k, v in sorted(label_group_mapping.items())},
    "train_loss": train_loss,
    "val_loss": val_result["loss"],
    "test_loss": test_result["loss"],
    "parameter_delta_l2": parameter_delta_l2,
    "source_manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "source_training_index_sha256": training_index_sha256,
    "source_preflight_commit": "351da8cec1aa41de1d5426b5d530b8c53a5af7fa",
    "warning": "Smoke test funcional. No utilizar como modelo final.",
}
torch.save(checkpoint, checkpoint_path)
save_prediction_preview(preview_path)

def artifact_hashes(paths: list[Path]) -> dict[str, str]:
    return {path.name: sha256_stream(path) for path in paths if path.exists()}

summary = {
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "training_index_sha256": training_index_sha256,
    "preflight_success_confirmed": True,
    "index_rows": int(len(spider_index_df)),
    "selected_pairs": int(len(selected_pairs_df)),
    "train_pairs": CFG.TRAIN_PAIRS,
    "val_pairs": CFG.VAL_PAIRS,
    "test_pairs": CFG.TEST_PAIRS,
    "downloaded_objects": downloaded_objects,
    "verified_downloads": verified_downloads,
    "train_slices": int((slice_records_df.split == "train").sum()),
    "val_slices": int((slice_records_df.split == "val").sum()),
    "test_slices": int((slice_records_df.split == "test").sum()),
    "device": str(device),
    "target_size": list(CFG.TARGET_SIZE),
    "num_classes": CFG.NUM_CLASSES,
    "base_channels": CFG.BASE_CHANNELS,
    "epochs_completed": CFG.MAX_EPOCHS,
    "train_batches": int(len(train_history)),
    "val_batches": int(val_result["batches"]),
    "test_batches": int(test_result["batches"]),
    "optimizer_steps": int(optimizer_steps),
    "train_loss": train_loss,
    "val_loss": float(val_result["loss"]),
    "test_loss": float(test_result["loss"]),
    "foreground_dice_validation": val_result["foreground_dice"],
    "foreground_dice_test": test_result["foreground_dice"],
    "prediction_min_class": int(min(val_result["prediction_min_class"], test_result["prediction_min_class"])),
    "prediction_max_class": int(max(val_result["prediction_max_class"], test_result["prediction_max_class"])),
    "parameter_delta_l2": parameter_delta_l2,
    "checkpoint_saved": checkpoint_path.exists(),
    "preview_saved": preview_path.exists(),
    "gcs_write_operations": GCS_WRITE_OPERATIONS,
}
summary["SMOKE_TEST_SUCCESS"] = bool(
    summary["preflight_success_confirmed"]
    and summary["index_rows"] == 447
    and training_index_sha256 == expected_index_sha
    and summary["selected_pairs"] == 6
    and selected_pairs_df["patient_id"].nunique() == 6
    and summary["downloaded_objects"] == 12
    and summary["verified_downloads"] == 12
    and summary["train_slices"] > 0
    and summary["val_slices"] > 0
    and summary["test_slices"] > 0
    and bool(slice_records_df.groupby("split")["has_foreground"].any().all())
    and CFG.MAX_EPOCHS == 1
    and optimizer_steps > 0
    and parameter_delta_l2 > 0
    and all(math.isfinite(float(v)) for v in [train_loss, val_result["loss"], test_result["loss"]])
    and summary["prediction_min_class"] >= 0
    and summary["prediction_max_class"] < CFG.NUM_CLASSES
    and summary["checkpoint_saved"]
    and summary["preview_saved"]
    and GCS_WRITE_OPERATIONS == 0
)
if not summary["SMOKE_TEST_SUCCESS"]:
    raise RuntimeError("Smoke test funcional no cumplio todas las condiciones estrictas")
with summary_path.open("w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2)
artifacts = [checkpoint_path, preview_path, selected_pairs_path, slice_records_path, batch_history_path, mapping_path, summary_path]
evidence = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "44_gcs_spider_smoke_test.ipynb",
    "repository_commit": REPOSITORY_COMMIT,
    "preflight_commit": "351da8cec1aa41de1d5426b5d530b8c53a5af7fa",
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "training_index_sha256": training_index_sha256,
    "artifact_sha256": artifact_hashes(artifacts),
    "gcs_read_only": GCS_WRITE_OPERATIONS == 0,
    "verified_downloads": verified_downloads,
}
with evidence_path.open("w", encoding="utf-8") as fh:
    json.dump(evidence, fh, indent=2)
print(json.dumps(summary, indent=2))